In [1]:
# Library
import requests
import pandas as pd
import numpy as np
import time
import json
import os
import getpass
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Folder
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/clean", exist_ok=True)
os.makedirs("../report", exist_ok=True)

# Settingan Visual untuk Plot 
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


In [2]:
# 1. Konfigurasi Github Token dan Target Repo
# Konfigurasi Input Token 
print("Masukkan GitHub Personal Access Token Anda:")
GITHUB_TOKEN = getpass.getpass()

# Konfigurasi target repo
REPO = "pandas-dev/pandas"
HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"}

print("\n Konfigurasi selesai.")
print(f"  Target repo : {REPO}")
print(f"  Token       : {'*' * 20}")

Masukkan GitHub Personal Access Token Anda:

 Konfigurasi selesai.
  Target repo : pandas-dev/pandas
  Token       : ********************


In [3]:
# 2. Data Fetching dari Github API untuk PR dan Issues
def fetch_github_data(endpoint, params, max_pages=20):
    all_data = []
    url = f"https://api.github.com/repos/{REPO}/{endpoint}"
    print(f"Mengambil data dari endpoint: /{endpoint}")
    
    for page in range(1, max_pages + 1):
        params["page"] = page
        params["per_page"] = 100
        
        response = requests.get(url, headers=HEADERS, params=params)
        
        if response.status_code != 200:
            print(f"Berhenti di halaman {page}. Status code: {response.status_code}")
            break
            
        data = response.json()
        if not data:
            print(f"Semua data sudah terambil.Selesai di halaman {page - 1}.")
            break 
            
        all_data.extend(data)
        print(f"Halaman {page}: {len(data)} item  |  Total terkumpul: {len(all_data)}")
        time.sleep(1) 

        print(f"\nSelesai! Total data dari /{endpoint}: {len(all_data)} item\n")

    return all_data

# A. Fetch Pull Requests (RQ1)
pr_params = {"state": "closed", "sort": "created", "direction": "desc"}
raw_prs = fetch_github_data("pulls", pr_params, max_pages=30) 
with open("../data/raw/raw_prs.json", "w") as f: json.dump(raw_prs, f) # Simpan data mentah sebagai backup
print(f"Data berhasil disimpan ke: {"../data/raw/raw_prs.json"}")
print(f"\nTotal raw PR tersedia: {len(raw_prs)}")

# B. Fetch Issues (RQ2 & RQ3)
issue_params = {"state": "closed", "sort": "created", "direction": "desc", "since": "2020-01-01T00:00:00Z"}
raw_issues = fetch_github_data("issues", issue_params, max_pages=40)
with open("../data/raw/raw_issues.json", "w") as f: json.dump(raw_issues, f) # Simpan data mentah sebagai backup
print(f"Data berhasil disimpan ke: {"../data/raw/raw_issues.json"}")
print(f"\nTotal raw Issues tersedia: {len(raw_issues)}")

print(f"\n Selesai! Total PR ditarik: {len(raw_prs)} | Total Issue ditarik: {len(raw_issues)}")

Mengambil data dari endpoint: /pulls
Halaman 1: 100 item  |  Total terkumpul: 100

Selesai! Total data dari /pulls: 100 item

Halaman 2: 100 item  |  Total terkumpul: 200

Selesai! Total data dari /pulls: 200 item

Halaman 3: 100 item  |  Total terkumpul: 300

Selesai! Total data dari /pulls: 300 item

Halaman 4: 100 item  |  Total terkumpul: 400

Selesai! Total data dari /pulls: 400 item

Halaman 5: 100 item  |  Total terkumpul: 500

Selesai! Total data dari /pulls: 500 item

Halaman 6: 100 item  |  Total terkumpul: 600

Selesai! Total data dari /pulls: 600 item

Halaman 7: 100 item  |  Total terkumpul: 700

Selesai! Total data dari /pulls: 700 item

Halaman 8: 100 item  |  Total terkumpul: 800

Selesai! Total data dari /pulls: 800 item

Halaman 9: 100 item  |  Total terkumpul: 900

Selesai! Total data dari /pulls: 900 item

Halaman 10: 100 item  |  Total terkumpul: 1000

Selesai! Total data dari /pulls: 1000 item

Halaman 11: 100 item  |  Total terkumpul: 1100

Selesai! Total data da

In [4]:
# 3. Data Cleaning untuk PR
PANDAS_V2_DATE = pd.Timestamp("2023-04-03")

pr_list               = []
jumlah_draft_dilewati = 0
jumlah_tanpa_closed   = 0
jumlah_anomali        = 0

for pr in raw_prs:
    if pr.get("draft") == True: 
        jumlah_draft_dilewati += 1
        continue 
    
    created   = pd.Timestamp(pr["created_at"])
    closed_at = pr.get("closed_at")
    merged_at = pr.get("merged_at")
    
    if closed_at is None:
        jumlah_tanpa_closed += 1
        continue

    closed = pd.Timestamp(closed_at)
    days   = (closed - created).total_seconds() / 86400

    if days <= 0:
        jumlah_anomali += 1
        continue

    is_merged = True if merged_at else False

    pr_list.append({
        "number"         : pr["number"],
        "created_at"     : created,
        "merged"         : is_merged,
        "outcome"        : 1 if is_merged else 0,  
        "resolution_days": round(days, 2)
    })

df_prs = pd.DataFrame(pr_list)

print("HASIL CLEANING DATA PULL REQUEST")
print(f"Raw PR masuk              : {len(raw_prs)}")
print(f"Draft PR dilewati         : {jumlah_draft_dilewati}")
print(f"PR tanpa closed_at        : {jumlah_tanpa_closed}")
print(f"Data anomali (days <= 0)  : {jumlah_anomali}")
print(f"───────────────────────────────")
print(f"PR valid (df_prs)         : {len(df_prs)}")
print(f"PR Merged   (k)           : {df_prs['merged'].sum()}")
print(f"PR Rejected (m)           : {(~df_prs['merged']).sum()}")
print()
df_prs.head()

HASIL CLEANING DATA PULL REQUEST
Raw PR masuk              : 3000
Draft PR dilewati         : 96
PR tanpa closed_at        : 0
Data anomali (days <= 0)  : 0
───────────────────────────────
PR valid (df_prs)         : 2904
PR Merged   (k)           : 2000
PR Rejected (m)           : 904



,number,created_at,merged,outcome,resolution_days
0,65743,2026-05-28 09:58:29+00:00,True,1,0.28
1,65736,2026-05-26 17:26:40+00:00,True,1,0.05
2,65732,2026-05-25 14:10:22+00:00,True,1,0.96
3,65731,2026-05-25 13:53:44+00:00,True,1,0.01
4,65729,2026-05-25 12:05:11+00:00,True,1,1.17


In [5]:
# 4. Data Cleaning untuk Issues
issue_list = []
jumlah_pr_dilewati  = 0
jumlah_anomali_iss  = 0

for issue in raw_issues:
    if "pull_request" in issue:
        jumlah_pr_dilewati += 1
        continue
    
    labels = [label["name"].lower() for label in issue.get("labels", [])]
    is_bug = any("bug" in label for label in labels)
    
    created = pd.Timestamp(issue["created_at"])
    closed_at = issue.get("closed_at")
    if closed_at is None: continue

    closed = pd.Timestamp(closed_at)
    days   = (closed - created).total_seconds() / 86400

    if days <= 0:
        jumlah_anomali_iss += 1
        continue

    issue_list.append({
        "number"         : issue["number"],
        "title"          : issue.get("title", ""),  
        "type"           : "bug" if is_bug else "other",
        "created_at"     : created,
        "resolution_days": round(days, 2)
    })

df_issues = pd.DataFrame(issue_list)

# Buat dataframe khusus bug per bulan untuk Poisson (RQ2)
df_issues["yearmonth"] = df_issues["created_at"].dt.to_period("M")
df_bugs = df_issues[df_issues["type"] == "bug"].copy()
monthly_bugs = df_bugs.groupby("yearmonth").size().reset_index(name="bug_count")
monthly_bugs["timestamp"] = monthly_bugs["yearmonth"].dt.to_timestamp()
monthly_bugs["period"]    = monthly_bugs["timestamp"].apply(
    lambda x: "post_v2" if x >= PANDAS_V2_DATE else "pre_v2"
)

print("HASIL CLEANING DATA ISSUES")
print(f"Raw issues masuk               : {len(raw_issues)}")
print(f"PR yang dilewati               : {jumlah_pr_dilewati}")
print(f"Anomali (days <= 0)            : {jumlah_anomali_iss}")
print(f"  ───────────────────────────────")
print(f"Issue valid (df_issues)        : {len(df_issues)}")
print(f"Tipe Bug                       : {len(df_bugs)}")
print(f"Tipe Lain                      : {len(df_issues) - len(df_bugs)}")
print(f"Bulan data bug (monthly_bugs)  : {len(monthly_bugs)} bulan")
print()
df_issues.head()

C:\Users\Darren CW\AppData\Local\Temp\ipykernel_15564\2707464164.py:36: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_issues["yearmonth"] = df_issues["created_at"].dt.to_period("M")


HASIL CLEANING DATA ISSUES
Raw issues masuk               : 4000
PR yang dilewati               : 3297
Anomali (days <= 0)            : 0
  ───────────────────────────────
Issue valid (df_issues)        : 703
Tipe Bug                       : 331
Tipe Lain                      : 372
Bulan data bug (monthly_bugs)  : 15 bulan



,number,title,type,created_at,resolution_days,yearmonth
0,65685,"BUG: Index.where(..., Index) fails on 3.1.0 bu...",bug,2026-05-19 13:39:44+00:00,2.04,2026-05
1,65684,ENH: Add `df.corrupt()` accessor to inject ran...,other,2026-05-19 06:52:43+00:00,0.55,2026-05
2,65553,ENH: Improve missing column KeyError message i...,other,2026-05-09 11:18:01+00:00,0.21,2026-05
3,65550,BUG: `groupby` cumprod/cummin/cummax returns ...,bug,2026-05-08 22:56:54+00:00,6.53,2026-05
4,65518,BUG: Conversion from np.ma.masked_array to pd....,bug,2026-05-08 01:11:36+00:00,0.26,2026-05


In [6]:
# 5. Simpan Dataset Bersih ke CSV
df_prs.to_csv("../data/clean/prs_clean.csv", index=False)
df_issues.to_csv("../data/clean/issues_full.csv", index=False)
monthly_bugs.to_csv("../data/clean/monthly_bugs.csv", index=False)

print("Data berhasil dibersihkan dan disimpan ke folder clean")
print(f"prs_clean.csv:{len(df_prs)} baris")
print(f"issues_full.csv:{len(df_issues)} baris")
print(f"monthly_bugs.csv:{len(monthly_bugs)} baris")

Data berhasil dibersihkan dan disimpan ke folder clean
prs_clean.csv:2904 baris
issues_full.csv:703 baris
monthly_bugs.csv:15 baris


In [ ]:
# . EXPLORATORY DATA ANALYSIS (EDA)
# Plot RQ1 
# Hitung statistik
k = df_prs["merged"].sum()
n = len(df_prs)
m = n - k

plt.figure(figsize=(10, 5))
sns.countplot(data=df_prs, x="merged", palette=["#EF5350", "#4CAF50"])
plt.title(f"Proporsi Outcome PR\nTotal PR: {n} | Merged (k): {k} | Rejected (m): {m}")
plt.xlabel("Apakah PR di-merge?")
plt.ylabel("Jumlah")
plt.show()

print(f"Preview Nilai MLE Bernoulli (θ̂) = {k/n:.4f}")

In [ ]:
# 4. EXPLORATORY DATA ANALYSIS (EDA)
# Plot RQ2
plt.figure(figsize=(12, 6))
# Convert period ke string agar mudah diplot dengan seaborn
monthly_bugs['timestamp'] = monthly_bugs['yearmonth'].dt.to_timestamp()
monthly_bugs = monthly_bugs.sort_values("timestamp")

sns.lineplot(data=monthly_bugs, x='timestamp', y='bug_count', hue='period', marker="o")
plt.axvline(x=PANDAS_V2_DATE, color="red", linestyle="--", label="Pandas 2.0 Rilis")

plt.xticks(rotation=45)
plt.title("Tren Jumlah Laporan Bug per Bulan (Pre vs Post Pandas 2.0)")
plt.xlabel("Bulan")
plt.ylabel("Jumlah Bug")
plt.legend()
plt.tight_layout()
plt.show()

# Preview rata-rata (Lambda Poisson)
pre_data = monthly_bugs[monthly_bugs['period'] == 'pre_v2']['bug_count']
post_data = monthly_bugs[monthly_bugs['period'] == 'post_v2']['bug_count']
lam_pre = monthly_bugs[monthly_bugs['period'] == 'pre_v2']['bug_count'].mean()
lam_post = monthly_bugs[monthly_bugs['period'] == 'post_v2']['bug_count'].mean()
print(f"Preview rata-rata bug sebelum (λ̂1) = {lam_pre:.2f}")
print(f"Preview rata-rata bug sesudah (λ̂2) = {lam_post:.2f}")

In [ ]:
# 4. EXPLORATORY DATA ANALYSIS (EDA)
# Plot RQ3
plt.figure(figsize=(10, 5))
sns.histplot(df_issues['resolution_days'], bins=50, kde=True, color="purple")
plt.axvline(30, color='red', linestyle='--', label='Batas 30 Hari')

plt.xlim(0, 100) # Potong visualisasi di 100 hari agar tidak melebar
plt.title("Distribusi Waktu Penyelesaian Issue (Resolution Time)")
plt.xlabel("Lama Penyelesaian (Hari)")
plt.ylabel("Frekuensi")
plt.legend()
plt.show()

# Hitung probabilitas empiris
lebih_dari_30 = (df_issues['resolution_days'] > 30).sum()
total_issue = len(df_issues)
print(f"Probabilitas empiris issue > 30 hari: {lebih_dari_30 / total_issue:.4f}")